In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("HW_3a_robot_arm.ipynb")

# Robot arm

Weeks 1 and part of week 2: Building the robot arm and forward kinematics

Slides: https://docs.google.com/presentation/d/17aiTBmPZidR6op7TvqYRzYatuc_NETYA1BhgpSHQ-FM/edit?usp=sharing

In [ ]:
# The usual imports
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# matrix_rouintes.py functions
import matrix_routines as mt

# This is the class you wrote/are writing in the lab
from arm_component import ArmComponent

# This is the class you are filling in for the homework
from robot_arm import RobotArm2D

In [ ]:
# These commands will force JN to actually re-load the external file when you re-execute the import command
%load_ext autoreload
%autoreload 2

# Week 1: Building an arm and doing forward kinematics

Two parts to this problem: The first is creating all of the components and storing them in a data structure (rather than having each one named individually, as was done in the lab). This way, you can have an arbitrary number of links in the arm.

The second part of this assignment is computing the pose matrices for each link, given the angles for each link.

GUIDES: Decide how you will store all of the arm components. The simplest approach is to use a list. You can also use a class or a dictionary. Your code should be able to handle an arbitrary number of links.


<!-- END QUESTION -->

## Week 1, Part 1: Put all the components together

Step 1: Create the link components based on a list of link/base sizes. This is the **__init__** method in **robot_arm.py**

Step 2: Fill in the **get_** methods in **robot_arm.py**

In [ ]:
# These are example inputs for the create_arm_geometry function. 
base_size_param = (0.5, 1.0)   # Height, width
link_sizes_param = [(0.5, 0.25), (0.3, 0.1), (0.2, 0.05)]

robot_arm = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param)

In [ ]:
# Correct shape matrices
mat_base_check = np.array([[0.5, 0.0, 0], [0.0, 0.25, 0.25], [0.0, 0.0, 1.0]])
mat_link1_check = np.array([[0.25, 0.0, 0.25], [0.0, 0.125, 0.0], [0.0, 0.0, 1.0]])
mat_link2_check = np.array([[0.15, 0.0, 0.15], [0.0, 0.05, 0.0], [0.0, 0.0, 1.0]])
mat_link3_check = np.array([[0.1, 0.0, 0.1], [0.0, 0.025, 0.0], [0.0, 0.0, 1.0]])


In [ ]:
# This checks the base shape matrix. Notice:
#   Using get_base to get the instance of the base matrix
#   Calling get_shape_matrix() on that instance to get the shape matrix
#   Doing the comparison using np.all(np.isclose())
assert np.all(np.isclose(robot_arm.get_base().get_shape_matrix(), mat_base_check))

# GUIDES: Use the get methods and the "check" matrices above to check the rest of your matrices and your get functions
# Note: The autograder calls these checks, once for the base, once for each link, and once each for the palm & fingers. 

In [ ]:
grader.check("build_arm")

### Week 1, part 2: Forward kinematics

In this step you'll fill in **set_link_angles**, which will set the pose matrices of every link **ArmComponent**.

(See homework slides) Each component has a "chain" of matrices that takes the link to the right location. This chain consists of alternating rotation and translation matrices. Each chain consists of all of the translations and rotations for the previous links plus a translation and rotation matrix for the current link. The gripperlocation has the longest chain - it consists of all of the rotations and translation for every link. Adding the gripper and fingers is extra credit (see LabHW_Extra_credit_gripper.ipynb).

One confusing thing is that the *last* matrix to be applied is the first one in the chain, reading from left to right in the code (see slides for why). 

There is one "special" matrix - this is the one that takes the first link to the top of the base, pointed up. This is the *last* matrix in the chain. 

Building the chain is the same for all of the links, so you can do this in a **for** loop. This loop is a bit tricky because you don't *quite* take the matrix from the previous link and just add the next rotation/translation pair in the chain. So your code will look like:

**matrix_chain =** special base matrix

- For the link, add a rotation by the link's angle then a translation in x by the **previous** link's length

- Add this rotation/translation pair to the chain in **matrix_chain**

If the **for** loop is challenging, just set each matrix in turn and then look for the pattern.

For full credit this has to work no matter what the link lengths are and how many links there are (hence the **for** loop)

Reminder that you should be using the **set_pose_matrix** method to set the matrices for the components (so it plots properly).

Second reminder: You should be using mt.make_xx_matrix routines to make the matrices.

Third reminder: You can store information (like link length) in the class.

In [ ]:
# EXAMPLE CODE
# Getting out where the base of the matrix is and what rotation to use.

# This is the matrix that transforms the base
#  The first item in the arm geometry list is the base
arm_component_base = robot_arm.get_base()
arm_component_no_transform = ArmComponent(name="Base No transform", color="grey", shape_to_use="wedge")

# Draw the wedge in its original location, with the point we're interested in (and direction)
pt_on_top = np.array([0, 1, 1]).transpose()
vec_on_top = np.array([0, 1, 0]).transpose()

fig, axs = plt.subplots(1, 2, figsize=(6, 3))
axs[0].set_title("Wedge with pt")
arm_component_no_transform.plot(axs[0], b_do_pose_matrix=False)
axs[0].plot(pt_on_top[0], pt_on_top[1], 'Xk')
axs[0].arrow(x=pt_on_top[0], y=pt_on_top[1], dx=vec_on_top[0], dy=vec_on_top[1], color="red")

# Multiply the point and the vector by the matrix base's matrix
pt_on_top_moved = arm_component_base.get_shape_matrix() @ pt_on_top
vec_on_top_moved = arm_component_base.get_shape_matrix() @ vec_on_top
axs[1].set_title("Wedge with pt moved")
arm_component_base.plot(axs[1], b_do_pose_matrix=False)
axs[1].plot(pt_on_top_moved[0], pt_on_top_moved[1], 'Xk')
axs[1].arrow(x=pt_on_top_moved[0], y=pt_on_top_moved[1], dx=vec_on_top_moved[0], dy=vec_on_top_moved[1], color="red")

# arctan2 gets the arc tangent of the y, x, and correctly handles the quadrants
angle_of_rotation = np.arctan2(vec_on_top_moved[1], vec_on_top_moved[0])

print(f"Angle rotated by {angle_of_rotation}")

In [ ]:
# Check the combined link rotations
# Several different angles to check your routines with 
# Pass the one you want to check into set_matrices_all_components in the cell below
#  Feel free to change these
angles_none = [0.0, 0.0, 0.0]
angles_check_link_0 = [np.pi/4, 0.0, 0.0]
angles_check_link_0_1 = [np.pi/4, -np.pi/4, 0.0]

# Don't change this one
angles_check = [np.pi/2, -np.pi/4, -3.0 * np.pi/4]


In [ ]:
# Use this cell to visually check the results
# With angles_none it should point straight up
# GUIDES: Change angles_check to be the angles above to check various angle combinations
robot_arm.set_link_angles(angles_check)

fig, axs = plt.subplots(1, 1, figsize=(6, 6))
robot_arm.plot(axs=axs)


In [ ]:
# Don't change this one - the cells below assume these are the angles
angles_check = [np.pi/2, -np.pi/4, -3.0 * np.pi/4]
robot_arm.set_link_angles(angles_check)


In [ ]:
# Check the returned values
np.set_printoptions(precision=4, floatmode='fixed')  # Print out with 4 digits of precision

mat_check_base = np.identity(3)
print(robot_arm.get_base().get_pose_matrix())
assert np.all(np.isclose(robot_arm.get_base().get_pose_matrix(), mat_check_base, atol=0.01))

In [ ]:
mat_check_link_1 = np.array([[ -1.0,  0.0,  0.0], \
                             [  0.0, -1.0,  0.5], \
                             [  0.0,  0.0,  1.0]])
                             
mat_check_link_2 = np.array([[ -0.7071, -0.7071, -0.5], \
                             [  0.7071, -0.7071,  0.5], \
                             [  0.0,     0.0,  1.0]])

mat_check_link_3 = np.array([[ 1.0, 0.0, -0.71213], \
                             [ 0.0, 1.0,  0.71213], \
                             [ 0.0, 0.0,  1.0]])

for ilink, m in enumerate((mat_check_link_1, mat_check_link_2, mat_check_link_3)):
    print(robot_arm.get_link(ilink).get_pose_matrix())
    assert(np.all(np.isclose(robot_arm.get_link(ilink).get_pose_matrix(), m, atol=0.01)))

In [ ]:
grader.check("forward_ik")

# Week 2: End of arm location

GUIDES: edit **get_end_location** to return the x,y location of the end of the last link.


In [ ]:
# Check the end location method
# As in the previous problem, you can use the "simpler" angles to check your function
angles_check_end_location = [np.pi/3, -np.pi/6, 3.0 * np.pi/6]

robot_arm_2 = RobotArm2D(base_arm_size=base_size_param, link_sizes=link_sizes_param)
robot_arm_2.set_link_angles(angles_check_end_location)

# Check the end location is correct (there is plotting code in the next cell)
end_loc = robot_arm_2.get_end_location()
assert np.isclose(end_loc[0], -0.7562, atol=0.01) and np.isclose(end_loc[1], 0.9098, atol=0.01)

In [ ]:

# Now actually plot - the grasp grip location should show up as a green cross
fig2, axs2 = plt.subplots(1, 1, figsize=(6, 6))
robot_arm_2.plot(axs2)


In [ ]:
# Do NOT change these or the autograder tests won't work
angles_check_end_location = [np.pi/3, -np.pi/6, 3.0 * np.pi/6]

# Actually set the matrices
robot_arm_2.set_link_angles(angles_check_end_location)

# Check the end location is correct (there is plotting code in the next cell)
end_loc_check = robot_arm_2.get_end_location()
print(f"{end_loc_check}")


In [ ]:
grader.check("gripper_loc")

## Generalization check

If nothing has been "hardwired" in, this should just work - changing the geometry and the angles. However, if you've hardwired something in, it probably won't...

In [ ]:
# Create another arm geometry
base_size_longer_param = (0.5, 0.25) # squished (height, width)
link_sizes_longer_param = [(0.3, 0.15), (0.2, 0.09), (0.1, 0.05), (0.075, 0.03)]
palm_width_longer_param = 0.15
finger_length_longer_param = (0.085, 0.015)


# Ths creates a different robot arm - one that is longer and with an additinal link
robot_arm_longer = RobotArm2D(base_size_longer_param, link_sizes_longer_param)

# Set the angles of the arm
angles_start_longer = [-np.pi/4.0, -np.pi/4, 1.2 * np.pi/4, -1 * np.pi/8]
robot_arm_longer.set_link_angles(angles_start_longer)

end_loc_longer_check = robot_arm_2.get_end_location()


assert np.all(np.isclose(end_loc_longer_check[0:2], (-0.756218, 0.909808), atol=0.01))

In [ ]:
# Now actually plot - the grasp grip location should show up as a green cross
fig3, axs3 = plt.subplots(1, 1, figsize=(6, 6))
robot_arm_longer.plot(axs3)
axs3.set_title("Longer arm")

In [ ]:
grader.check("generalization_check")

## Hours and collaborators
Required for every assignment - fill out before you hand-in.

Listing names and websites helps you to document who you worked with and what internet help you received in the case of any plagiarism issues. You should list names of anyone (in class or not) who has substantially helped you with an assignment - or anyone you have *helped*. You do not need to list TAs.

Listing hours helps us track if the assignments are too long.

In [ ]:
import os

# List of names (creates a set)
worked_with_names = {"not filled out"}
# List of URLS FA26 (creates a set)
websites = {"not filled out"}
# Approximate number of hours, including lab/in-class time
hours = -1.5

# VS Code stores the path in a special global variable
if '__vsc_ipynb_file__' in globals():
    notebook_path = globals()['__vsc_ipynb_file__']
    notebook_name = os.path.basename(notebook_path)
    json_name = notebook_name[:-6] + "_source.json"
    if not os.path.exists(json_name):
        print(f"Could not find the json file {json_name}; make sure the required VSCode extension is installed and running")


In [ ]:
grader.check("hours_collaborators")

### To submit

(Did you read me?)

- Submit this .ipynb file, the .json file, AND arm_component.py AND robot_arm.py. If you don't include arm_component.py or robot_arm.py Gradescope cannot magically reach out to your computer and find it.
- We will supply matrix_routines.py for you (it won't break anything if you do include it).
- As always, do a restart-runall before submitting and make sure your plots are visible.
- Do NOT include the other .ipynb files.

If the Gradescope autograder fails, please check here first for common reasons for it to fail
    https://docs.google.com/presentation/d/1tYa5oycUiG4YhXUq5vHvPOpWJ4k_xUPp2rUNIL7Q9RI/edit?usp=sharing

Lots of people forget arm_component.py or robot_arm.py. Or include another.ipynb file. Please check your autograder score to see if you are one of them.

Make sure you remove all the print statements you put in that print out lots of stuff.